# Indian Sign Language

See the project README for data, environment, evaluation, and known limitations. This notebook records exploratory work; outputs are cleared for a fresh run.


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')


In [ ]:
import numpy as np
from time import time
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import cv2
import sklearn
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import utils
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn import metrics
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import load_img, img_to_array, array_to_img



In [ ]:
Indian = '/content/gdrive/MyDrive/Indian'

In [ ]:
img = cv2.imread('/content/gdrive/MyDrive/Indian/P/0.jpg',0)
plt.imshow(img,cmap="gray")

In [ ]:
#Gaussian-Blur
def edge_detection(image):
  Picking_Img = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
  image = cv2.cvtColor(Picking_Img,cv2.COLOR_RGB2GRAY)
  minValue = 70
  blur = cv2.GaussianBlur(image,(5,5),2)
  th3 = cv2.adaptiveThreshold(blur,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,11,2)
  ret, edge = cv2.threshold(th3, minValue, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
  return edge
img = cv2.imread('/content/gdrive/MyDrive/Indian/P/0.jpg')
img = edge_detection(img)
img = cv2.resize(img, (128, 128))
plt.imshow(img,cmap="gray")

In [ ]:
#Canny-Edge-Detection
def canny(image):
  Picking_Img = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
  Gray_Img = cv2.cvtColor(Picking_Img,cv2.COLOR_RGB2GRAY)
  def threshold_vision(img_path):
    _,threshold_Img = cv2.threshold(Gray_Img,90,255,cv2.THRESH_BINARY_INV)
    return threshold_Img
  Threshold_Img = threshold_vision(image)
  canny_img = cv2.Canny(Threshold_Img,10,100)
  return(canny_img)
img = cv2.imread('/content/gdrive/MyDrive/Indian/P/0.jpg') 
img = canny(img)
img = cv2.resize(img, (128, 128))
plt.imshow(img,cmap="gray")

In [ ]:
#k-means
def kmeans(image):
  pixel_vals = image.reshape((-1,3))
  pixel_vals = np.float32(pixel_vals)
  criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.85)
  k = 2
  retval, labels, centers = cv2.kmeans(pixel_vals, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
  centers = np.uint8(centers)
  segmented_data = centers[labels.flatten()]
  segmented_image = segmented_data.reshape((image.shape))
  return(segmented_image)
img = cv2.imread('/content/gdrive/MyDrive/Indian/P/0.jpg') 
img = kmeans(img) 
img = cv2.resize(img, (128, 128))
plt.imshow(img,cmap="gray")


In [ ]:
img = cv2.imread('/content/gdrive/MyDrive/Indian/P/0.jpg') 
img = kmeans(img)
img = canny(img)
img = cv2.resize(img, (128,128))
plt.imshow(img,cmap="gray")

In [ ]:
from tensorflow.keras.utils import to_categorical

In [ ]:
def load_images():
    images = []
    labels = []
    index = -1
    folders = sorted(os.listdir(Indian))
    for folder in folders:
        index += 1
        print("Loading images from folder ", folder ," has started.")
        for image in os.listdir(Indian + '/' + folder):
            img = cv2.imread(Indian + '/' + folder + '/' + image)
            img = kmeans(img)
            img = canny(img)
            img = cv2.resize(img, (64, 64)) 
            images.append(img)
            labels.append(index)

    images = np.array(images)
    images = images.astype('float32')/255.0
    labels = to_categorical(labels)

    x_train, x_test, y_train, y_test = train_test_split(images, labels, test_size=0.1, random_state=42, stratify=np.argmax(labels, axis=1))

    return x_train, x_test, y_train, y_test


In [ ]:
start = time()
x_train, x_test, y_train, y_test = load_images()
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42, stratify=np.argmax(y_train, axis=1)
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers, models
from keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from keras.losses import CategoricalCrossentropy
import tensorflow as tf

In [ ]:
base_model = Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64,1)),
    layers.MaxPool2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPool2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPool2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(35, activation='softmax')
])
base_model.summary()


In [ ]:

epochs = 10
learning_rate = 0.0001
adam = Adam(lr=learning_rate)
base_model.compile(optimizer=adam, loss='categorical_crossentropy', metrics=['accuracy'])
history = base_model.fit(
                    x_train, y_train,
                    epochs=epochs,
                    verbose=1,
                    validation_data=(x_valid, y_valid), shuffle=True)
bptime=time()-start




In [ ]:
base_model.save('/content/gdrive/MyDrive/models/basepapermodel.h5')

In [ ]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,confusion_matrix,roc_curve,roc_auc_score

In [ ]:
map_characters = {0: '1', 1: '2', 2: '3', 3: '4', 4: '5', 5: '6', 6: '7', 7: '8', 8: '9', 9: 'A', 10: 'B', 11: 'C', 12: 'D',
                  13: 'E', 14: 'F', 15: 'G', 16: 'H', 17: 'I', 18: 'J', 19: 'K', 20: 'L', 21: 'M', 22: 'N', 23: 'O', 24: 'P',
                  25: 'Q', 26: 'R', 27: 'S', 28: 'T', 29: 'U', 30: 'V', 31: 'W', 32: 'X', 33: 'Y', 34: 'Z'}

In [ ]:
class_labels = list(map_characters.values())

In [ ]:
score = base_model.evaluate(x_test,y_test, verbose=0)
print('\nKeras CNN - accuracy:', score[1], '\n')

y_pred = base_model.predict(x_test, verbose = 1)
Y_pred_classes = np.argmax(y_pred,axis = 1) 
Y_true = np.argmax(y_test,axis = 1) 
report = metrics.classification_report(Y_true, Y_pred_classes, target_names=class_labels)
print(report)


In [ ]:
input=[]
check = cv2.imread('/content/gdrive/MyDrive/Indian/1/0.jpg')
check = kmeans(check)
check = canny(check)

check = cv2.resize(check, (64, 64))

input.append(check)
input = np.array(input)
input = input.astype('float32')/255.0


In [ ]:
base_model.predict(input)


In [ ]:
image1=cv2.imread('/content/gdrive/MyDrive/Indian/1/0.jpg',0)
plt.imshow(image1)

In [ ]:
input=[]
check = cv2.imread('/content/gdrive/MyDrive/Indian/P/0.jpg')
check = kmeans(check)
check = canny(check)
check = cv2.resize(check, (64, 64))

input.append(check)
input = np.array(input)
input = input.astype('float32')/255.0
base_model.predict(input)


In [ ]:
image1=cv2.imread('/content/gdrive/MyDrive/Indian/P/0.jpg',0)
plt.imshow(image1)

####Training Different Architectures


In [ ]:
train_folder = '/content/gdrive/MyDrive/Indian'
all_data = []
for folder in os.listdir(train_folder):
    
    label_folder = os.path.join(train_folder, folder)
    onlyfiles = [{'label':folder,'path':os.path.join(label_folder, f)} for f in os.listdir(label_folder) if os.path.isfile(os.path.join(label_folder, f))]
    #print(onlyfiles)
    all_data += onlyfiles
data_df = pd.DataFrame(all_data)
data_df

In [ ]:
x_train,x_holdout = train_test_split(data_df, test_size= 0.10, random_state=42,stratify=data_df[['label']])
x_train,x_test = train_test_split(x_train, test_size= 0.20, random_state=42,stratify=x_train[['label']])

In [ ]:
img_width, img_height = 75, 75
batch_size = 128
y_col = 'label'
x_col = 'path'
no_of_classes = len(data_df[y_col].unique())


train_datagen = ImageDataGenerator(rescale = 1/255.0)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=x_train,x_col=x_col, y_col=y_col,color_mode="rgb",
    target_size=(img_width, img_height),class_mode='categorical', batch_size=batch_size,
    shuffle=False,
)

validation_datagen = ImageDataGenerator(rescale = 1/255.0)
validation_generator = validation_datagen.flow_from_dataframe(
    dataframe=x_test, x_col=x_col, y_col=y_col,color_mode="rgb",
    target_size=(img_width, img_height), class_mode='categorical', batch_size=batch_size,
    shuffle=False
)

holdout_datagen = ImageDataGenerator(rescale = 1/255.0)
holdout_generator = holdout_datagen.flow_from_dataframe(
    dataframe=x_holdout, x_col=x_col, y_col=y_col,color_mode="rgb",
    target_size=(img_width, img_height), class_mode='categorical', batch_size=batch_size,
    shuffle=False
)

In [ ]:
from tensorflow.keras.applications import ResNet50V2


In [ ]:
resnet50v2arch= ResNet50V2(input_shape=(75, 75, 3), include_top=False, weights="imagenet")
resnet50v2arch.trainable = False 

In [ ]:
resnet50v2arch.summary()

In [ ]:
flatten_layer = layers.Flatten()
dense_layer_1 = layers.Dense(512, activation='relu')
dropout_layer_1 = layers.Dropout(0.5)
prediction_layer = layers.Dense(35, activation='softmax')


res_model = models.Sequential([
    resnet50v2arch,
    flatten_layer,
    dense_layer_1,
    dropout_layer_1,
    prediction_layer
])


In [ ]:
res_model.summary()

In [ ]:
start = time()
classes = 35
epochs = 10
learning_rate = 0.0001
adam = Adam(lr=learning_rate)
res_model.compile(optimizer=adam, loss='categorical_crossentropy', metrics=['accuracy'])
history = res_model.fit(train_generator,
                    epochs=epochs,
                    verbose=1,
                    validation_data=validation_generator, shuffle=True)
restime=time()-start

In [ ]:
res_model.save('/content/gdrive/MyDrive/models/resnet50v2model.h5')

In [ ]:
from keras.applications.vgg16 import VGG16


In [ ]:
VGG16arch = VGG16(weights = "imagenet", include_top = False, input_shape = (75, 75, 3))
VGG16arch.trainable = False 

In [ ]:
VGG16arch.summary()

In [ ]:
flatten_layer = layers.Flatten()
dense_layer_1 = layers.Dense(512, activation='relu')
dropout_layer_1 = layers.Dropout(0.5)
#dense_layer_2 = layers.Dense(512, activation='relu')
#dropout_layer_2 = layers.Dropout(0.5)
prediction_layer = layers.Dense(35, activation='softmax')


vgg16_model = models.Sequential([
    VGG16arch,
    flatten_layer,
    dense_layer_1,
    dropout_layer_1,
    #dense_layer_2,
    #dropout_layer_2,
    prediction_layer
])
vgg16_model.summary()

In [ ]:
start = time()
classes = 35
epochs = 10
learning_rate = 0.0001

adam = Adam(lr=learning_rate)
vgg16_model.compile(optimizer=adam, loss='categorical_crossentropy', metrics=['accuracy'])
history = vgg16_model.fit(train_generator,
                    epochs=epochs,
                    verbose=1,
                    validation_data=validation_generator, shuffle=True)
vgg_time=time()-start

In [ ]:
vgg16_model.save('/content/gdrive/MyDrive/models/vgg16model.h5')

In [ ]:
from tensorflow.keras.applications.inception_v3 import InceptionV3

In [ ]:
inceptionv3arch = InceptionV3(input_shape=(75,75,3), weights='imagenet', include_top=False)
inceptionv3arch.trainable = False

In [ ]:
inceptionv3arch.summary()

In [ ]:
flatten_layer = layers.Flatten()
dense_layer_1 = layers.Dense(512, activation='relu')
dropout_layer_1 = layers.Dropout(0.5)
#dense_layer_2 = layers.Dense(512, activation='relu')
#dropout_layer_2 = layers.Dropout(0.5)
prediction_layer = layers.Dense(35, activation='softmax')


incept_model = models.Sequential([
    inceptionv3arch,
    flatten_layer,
    dense_layer_1,
    dropout_layer_1,
    #dense_layer_2,
    #dropout_layer_2,
    prediction_layer
])
incept_model.summary()

In [ ]:
start = time()
classes = 35
epochs = 10
learning_rate = 0.0001

adam = Adam(lr=learning_rate)
incept_model.compile(optimizer=adam, loss='categorical_crossentropy', metrics=['accuracy'])
history = incept_model.fit(train_generator,
                    epochs=epochs,
                    verbose=1,
                    validation_data=validation_generator, 
                    shuffle=True)
incept_time=time()-start

In [ ]:
incept_model.save('/content/gdrive/MyDrive/models/inceptmodel.h5')

In [ ]:
print("Time Taken to Train CNN model : " , bptime)
print("Time Taken to Train CNN model with ResNet50v2 Architecture : " , restime)
print("Time Taken to Train CNN model with VGG16 Architecture : " , vgg_time)
print("Time Taken to Train CNN model with ResNet50v2 Architecture : " , incept_time)

In [ ]:
from tensorflow.keras.models import load_model

In [ ]:
inc=load_model('/content/gdrive/MyDrive/models/inceptmodel.h5')


In [ ]:
incpredictions = inc.predict(holdout_generator, verbose=1)
incpredicted_classes = np.argmax(incpredictions, axis=-1)
incpredictions = np.argmax(incpredictions, axis=-1)

inctrue_classes = holdout_generator.classes
incclass_labels = list(holdout_generator.class_indices.keys())  
report = metrics.classification_report(inctrue_classes, incpredicted_classes, target_names=class_labels)
print(report)

In [ ]:
incconfusion = metrics.confusion_matrix(inctrue_classes, incpredicted_classes)
print('Confusion Matrix\n')
print(incconfusion)

In [ ]:
res=load_model('/content/gdrive/MyDrive/models/resnet50v2model.h5')
respredictions = res.predict(holdout_generator, verbose=1)
respredicted_classes = np.argmax(respredictions, axis=-1)
respredictions = np.argmax(respredictions, axis=-1) #multiple categories

restrue_classes = holdout_generator.classes
resclass_labels = list(holdout_generator.class_indices.keys())  
report = metrics.classification_report(restrue_classes, respredicted_classes, target_names=class_labels)
print(report)

In [ ]:
resconfusion = metrics.confusion_matrix(restrue_classes, respredicted_classes)
print('Confusion Matrix\n')
print(resconfusion)

In [ ]:
vgg=load_model('/content/gdrive/MyDrive/models/vgg16model.h5')

In [ ]:
vgg=load_model('/content/gdrive/MyDrive/models/resnet50v2model.h5')
vgpredictions = vgg.predict(holdout_generator, verbose=1)
vgpredicted_classes = np.argmax(vgpredictions, axis=-1)
vgpredictions = np.argmax(vgpredictions, axis=-1) #multiple categories

vgtrue_classes = holdout_generator.classes
vgclass_labels = list(holdout_generator.class_indices.keys())  
report = metrics.classification_report(vgtrue_classes, vgpredicted_classes, target_names=class_labels)
print(report)

In [ ]:
vgconfusion = metrics.confusion_matrix(vgtrue_classes, vgpredicted_classes)
print('Confusion Matrix\n')
print(vgconfusion)

In [ ]:
bp=load_model('/content/gdrive/MyDrive/models/basepapermodel.h5')

In [ ]:
map_characters = {0: '1', 1: '2', 2: '3', 3: '4', 4: '5', 5: '6', 6: '7', 7: '8', 8: '9', 9: 'A', 10: 'B', 11: 'C', 12: 'D',
                  13: 'E', 14: 'F', 15: 'G', 16: 'H', 17: 'I', 18: 'J', 19: 'K', 20: 'L', 21: 'M', 22: 'N', 23: 'O', 24: 'P',
                  25: 'Q', 26: 'R', 27: 'S', 28: 'T', 29: 'U', 30: 'V', 31: 'W', 32: 'X', 33: 'Y', 34: 'Z'}
def incdemonstrate(path): 
  img=image.load_img(path,target_size=(75,75))
  x=image.img_to_array(img)
  x=np.array([x])
  x=x/255
  b=np.argmax(inc.predict(x))
  final=map_characters.get(b)
  return final
def resdemonstrate(path): 
  img=image.load_img(path,target_size=(75,75))
  x=image.img_to_array(img)
  x=np.array([x])
  x=x/255
  b=np.argmax(res.predict(x))
  final=map_characters.get(b)
  return final
def vggdemonstrate(path): 
  img=image.load_img(path,target_size=(75,75))
  x=image.img_to_array(img)
  x=np.array([x])
  x=x/255
  b=np.argmax(vgg.predict(x))
  final=map_characters.get(b)
  return final
def bpdemonstrate(path):
  input=[]
  check = cv2.imread(path)
  check = kmeans(check)
  check = canny(check)
  check = cv2.resize(check, (64, 64))
  input.append(check)
  input = np.array(input)
  input = input.astype('float32')/255.0
  b=np.argmax(bp.predict(input))
  final=map_characters.get(b)
  return final

In [ ]:
print(incdemonstrate('/content/gdrive/MyDrive/Indian/1/0.jpg'))
print(resdemonstrate('/content/gdrive/MyDrive/Indian/D/1002.jpg'))
print(vggdemonstrate('/content/gdrive/MyDrive/Indian/4/166.jpg'))
print(bpdemonstrate('/content/gdrive/MyDrive/Indian/6/1004.jpg'))